# 04 — Summary & Scorecard
**Objective:** pull every test-case result of **this run** into an O1–O3 objective scorecard.

**Customer sees:** four possible states per objective — PASS (green), FAIL (red), INCONCLUSIVE (amber: the evidence does not decide) and INCOMPLETE (grey: a test case of this run is missing).

## Step 1 — Results of the current run

In [ ]:
import pandas as pd
from IPython.display import display
from poc import results, viz

RID = results.run_id()
allr = results.load_run(RID)
print("run id:", RID, "|", len(allr), "test-case results in this run")

## Step 2 — Objective scorecard
O1 Spark↔S3, O2 Throttler, O3 S3 Tables — each mapped to its test cases. Results from other runs are ignored (listed as stale).

In [ ]:
sc = pd.DataFrame(results.scorecard(RID))
display(sc[["objective", "description", "status", "test_cases", "pass", "fail",
            "inconclusive", "missing", "not_pass"]]
        .style.apply(viz.status_style, axis=1))

**Presenter notes** (hidden)

Rule: any missing test case -> INCOMPLETE; else any FAIL -> FAIL; else any INCONCLUSIVE -> INCONCLUSIVE; else PASS.

## Step 3 — Test cases
Every test case of this run with its status and reason.

In [ ]:
tcs = pd.DataFrame([{"tc": tc, "name": r["name"], "status": results.status_of(r),
                     "detail": r.get("detail") or ""}
                    for tc, r in sorted(allr.items(),
                                        key=lambda kv: [float(x) for x in kv[0][2:].split(".")])])
display(tcs.style.apply(viz.status_style, axis=1))

## Step 4 — Key metrics
Headline numbers pulled from the raw results.

In [ ]:
kv = []
for tc in ("TC1.6", "TC2.4", "TC2.5", "TC2.6", "TC2.7", "TC3.7"):
    r = allr.get(tc)
    if r:
        kv.append({"tc": tc, **{k: v for k, v in r["metrics"].items()
                                if isinstance(v, (int, float, str))}})
viz.table(pd.DataFrame(kv).set_index("tc").T.fillna(""))

## Step 5 — Overall
`reports/PoC_Report.html` (scripts/report.py) carries the same scorecard for the same run ID.

In [ ]:
overall = results.overall(RID)
viz.banner(overall, "all objectives",
           f"run {RID} — see reports/PoC_Report.html")